<a href="https://colab.research.google.com/github/Saputoa21/Applied_ML_Spoiler_Detection_Group_Project_2025/blob/main/Transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transformers

Models used:

* Bert base cased(Arina): https://huggingface.co/google-bert/bert-base-cased
* RoBERTa base (Milica): https://huggingface.co/FacebookAI/roberta-base
* Distilbert base cased (Anastasiya): https://huggingface.co/distilbert/distilbert-base-cased

## Loading the Model

In [ ]:
!pip install transformers
!pip install datasets
!pip install evaluate
# !pip install transformers torch

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
print(torch.__version__)

2.6.0+cu124


In [ ]:
from transformers import DistilBertTokenizer, DistilBertModel

In [ ]:
# from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [ ]:
# Loading the model directly
distilbert_tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-cased')

distilbert_model = DistilBertModel.from_pretrained("distilbert-base-cased")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/263M [00:00<?, ?B/s]

In [ ]:
print(distilbert_tokenizer)

DistilBertTokenizer(name_or_path='distilbert-base-cased', vocab_size=28996, model_max_length=512, is_fast=False, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=True, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)


In [ ]:
input_str = "Spoiler alert!"

input_tokens = distilbert_tokenizer.tokenize(input_str)
print(f"Tokens of the input sequence: {input_tokens}")

input_ids = distilbert_tokenizer.convert_tokens_to_ids(input_tokens)
print(f"IDs assigned to the intput sequence: {input_ids}")

decoded = distilbert_tokenizer.decode(input_ids)
print(decoded)

model_inputs = distilbert_tokenizer("Spoiler alert!", return_tensors="pt")
print(model_inputs)

Tokens of the input sequence: ['S', '##po', '##iler', 'alert', '!']
IDs assigned to the intput sequence: [156, 5674, 25614, 10427, 106]
Spoiler alert!
{'input_ids': tensor([[  101,   156,  5674, 25614, 10427,   106,   102]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1]])}


## Loading Metrics

In [ ]:
import evaluate

accuracy_metric = evaluate.load("accuracy")
precision_metric = evaluate.load("precision")
f1_metric = evaluate.load("f1")
recall_metric = evaluate.load('recall')

matthews_metric = evaluate.load("matthews_correlation")

In [ ]:
print(accuracy_metric.description)


Accuracy is the proportion of correct predictions among the total number of cases processed. It can be computed with:
Accuracy = (TP + TN) / (TP + TN + FP + FN)
 Where:
TP: True positive
TN: True negative
FP: False positive
FN: False negative



In [ ]:
print(precision_metric.description)


Precision is the fraction of correctly labeled positive examples out of all of the examples that were labeled as positive. It is computed via the equation:
Precision = TP / (TP + FP)
where TP is the True positives (i.e. the examples correctly labeled as positive) and FP is the False positive examples (i.e. the examples incorrectly labeled as positive).



In [ ]:
print(f1_metric.description)


The F1 score is the harmonic mean of the precision and recall. It can be computed with the equation:
F1 = 2 * (precision * recall) / (precision + recall)



In [ ]:
print(recall_metric.description)


Recall is the fraction of the positive examples that were correctly labeled by the model as positive. It can be computed with the equation:
Recall = TP / (TP + FN)
Where TP is the true positives and FN is the false negatives.



In [ ]:
print(matthews_metric.description)


Compute the Matthews correlation coefficient (MCC)

The Matthews correlation coefficient is used in machine learning as a
measure of the quality of binary and multiclass classifications. It takes
into account true and false positives and negatives and is generally
regarded as a balanced measure which can be used even if the classes are of
very different sizes. The MCC is in essence a correlation coefficient value
between -1 and +1. A coefficient of +1 represents a perfect prediction, 0
an average random prediction and -1 an inverse prediction.  The statistic
is also known as the phi coefficient. [source: Wikipedia]



## Loading the dataset

In [54]:
from datasets import load_dataset, DatasetDict, Dataset
from transformers import DataCollatorWithPadding

In [51]:
#do so, if you mange to upload the file directly
# combined_df_exploded = load_dataset("combined_df_exploded.csv")

In [52]:
# do so, if you have your file as zip
# import zipfile
# import os

# zip_path = "/content/combined_df_exploded.zip"
# extract_path = "/content/unzipped"

# # Create a directory to extract if it doesn't exist
# os.makedirs(extract_path, exist_ok=True)

# # Extract the ZIP file
# with zipfile.ZipFile(zip_path, 'r') as zip_ref:
#     zip_ref.extractall(extract_path)

# # List extracted files
# !ls /content/unzipped

In [43]:
#do so if you have your file in google drive
import pandas as pd

csv_path = '/content/drive/MyDrive/Colab Notebooks/combined_df_exploded_1.csv'  # Adjust this path
combined_df_exploded = pd.read_csv(csv_path)

# Preview the data
combined_df_exploded.head()

,review_date,movie_id,user_id,is_spoiler,review_text,rating_x,review_summary,plot_summary,duration,genre,rating_y,release_date,plot_synopsis
0,10 February 2006,tt0111161,ur1898687,True,"In its Oscar year, Shawshank Redemption (writt...",10,A classic piece of unforgettable film-making.,Chronicles the experiences of a formerly succe...,2h 22min,Crime,9.3,1994-10-14,"In 1947, Andy Dufresne (Tim Robbins), a banker..."
1,10 February 2006,tt0111161,ur1898687,True,"In its Oscar year, Shawshank Redemption (writt...",10,A classic piece of unforgettable film-making.,Chronicles the experiences of a formerly succe...,2h 22min,Drama,9.3,1994-10-14,"In 1947, Andy Dufresne (Tim Robbins), a banker..."
2,6 September 2000,tt0111161,ur0842118,True,The Shawshank Redemption is without a doubt on...,10,Simply amazing. The best film of the 90's.,Chronicles the experiences of a formerly succe...,2h 22min,Crime,9.3,1994-10-14,"In 1947, Andy Dufresne (Tim Robbins), a banker..."
3,6 September 2000,tt0111161,ur0842118,True,The Shawshank Redemption is without a doubt on...,10,Simply amazing. The best film of the 90's.,Chronicles the experiences of a formerly succe...,2h 22min,Drama,9.3,1994-10-14,"In 1947, Andy Dufresne (Tim Robbins), a banker..."
4,3 August 2001,tt0111161,ur1285640,True,I believe that this film is the best story eve...,8,The best story ever told on film,Chronicles the experiences of a formerly succe...,2h 22min,Crime,9.3,1994-10-14,"In 1947, Andy Dufresne (Tim Robbins), a banker..."


In [50]:
#for training we need to remove unnecessary columns from the dataset
combined_df_exploded_cleaned = combined_df_exploded[['review_text', 'is_spoiler']]

combined_df_exploded_cleaned.head()

,review_text,is_spoiler
0,"In its Oscar year, Shawshank Redemption (writt...",True
1,"In its Oscar year, Shawshank Redemption (writt...",True
2,The Shawshank Redemption is without a doubt on...,True
3,The Shawshank Redemption is without a doubt on...,True
4,I believe that this film is the best story eve...,True


## Splitting the dataset

In [57]:
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
import pandas as pd

# 1. Start with your cleaned DataFrame
df = combined_df_exploded_cleaned  # has 'review_text' and 'is_spoiler'

# 2. Split into train/val/test manually (80/10/10)
train_val, test = train_test_split(df, test_size=0.1, random_state=42)
train, val = train_test_split(train_val, test_size=0.1111, random_state=42)  # 0.1111 of 0.9 ≈ 0.1

# 3. Convert each to Hugging Face Datasets
spoiler_dataset = DatasetDict({
    "train": Dataset.from_pandas(train.reset_index(drop=True)),
    "validation": Dataset.from_pandas(val.reset_index(drop=True)),
    "test": Dataset.from_pandas(test.reset_index(drop=True))
})

# 4. Truncate function — limit to first 50 tokens (words, not subwords yet)
def truncate(example):
    return {
        "text": " ".join(example["review_text"].split()[:50]),
        "label": example["is_spoiler"]
    }

# 5. Apply truncation
spoiler_dataset = spoiler_dataset.map(truncate)

# 6. Tokenization (using DistilBERT tokenizer)
from transformers import DistilBertTokenizerFast, DataCollatorWithPadding

distilbert_tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def tokenize_function(example):
    return distilbert_tokenizer(example["text"], padding=True, truncation=True)

# 7. Apply tokenizer
small_tokenized_dataset = spoiler_dataset.map(tokenize_function, batched=True, batch_size=16)

# 8. Prepare data collator
data_collator = DataCollatorWithPadding(tokenizer=distilbert_tokenizer)

Map:   0%|          | 0/164244 [00:00<?, ? examples/s]

Map:   0%|          | 0/20529 [00:00<?, ? examples/s]

Map:   0%|          | 0/20531 [00:00<?, ? examples/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

Map:   0%|          | 0/164244 [00:00<?, ? examples/s]

Map:   0%|          | 0/20529 [00:00<?, ? examples/s]

Map:   0%|          | 0/20531 [00:00<?, ? examples/s]

In [ ]:
from datasets import Dataset
from transformers import DistilBertTokenizerFast, DataCollatorWithPadding

# Convert pandas DataFrame to Hugging Face Dataset
spoiler_dataset = Dataset.from_pandas(combined_df_exploded_cleaned)

# Truncate to first 50 words
def truncate(example):
    return {
        "text": " ".join(example["review_text"].split()[:50]),
        "label": example["is_spoiler"]
    }

# Apply truncation
spoiler_dataset = spoiler_dataset.map(truncate)

# Load tokenizer
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

# Tokenization
def tokenize_function(example):
    return tokenizer(example["text"], padding=True, truncation=True)

# Tokenize the dataset
tokenized_spoiler_dataset = spoiler_dataset.map(tokenize_function, batched=True)

# Optional: Data collator for use in training/dataloader
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [56]:
spoiler_dataset = Dataset.from_pandas(combined_df_exploded_cleaned)

# Just take the first 50 tokens for speed on CPU
def truncate(example):
    return {
        'text': " ".join(example['review_text'].split()[:100]),
        'label': example['is_spoiler']
    }

# Take 128 random examples for train and 32 validation
small_imdb_dataset = DatasetDict(
    train=spoiler_dataset['train'].shuffle(seed=24).select(range(128)).map(truncate),
    val=spoiler_dataset['train'].shuffle(seed=24).select(range(128, 160)).map(truncate),
    test=spoiler_dataset['test'].shuffle(seed=24).select(range(128, 160)).map(truncate),
)

def tokenize_function(examples):
    return distilbert_tokenizer(examples["text"], padding=True, truncation=True)

small_tokenized_dataset = small_imdb_dataset.map(tokenize_function, batched=True, batch_size=16)
data_collator = DataCollatorWithPadding(tokenizer=distilbert_tokenizer)

KeyError: "Column train not in the dataset. Current columns in the dataset: ['review_text', 'is_spoiler']"

## Training

In [ ]:
import numpy as np
from transformers import TrainingArguments, Trainer
from transformers import AutoModelForSequenceClassification

# distilbert_model = DistilBertModel.from_pretrained("distilbert-base-cased", num_labels=2)

model = AutoModelForSequenceClassification.from_pretrained('distilbert/distilbert-base-cased', num_labels=2)


arguments = TrainingArguments(
    output_dir="/content/drive/MyDrive/Colab Notebooks/sample_cl_trainer", #change it if you want to store models somewhere else
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    logging_steps=8,
    num_train_epochs=5,
    eval_strategy="epoch", # run validation at the end of each epoch
    save_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    report_to='none',
    seed=224
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels),
    precision_metric.compute(predictions=predictions, references=labels),
    f1_metric.compute(predictions=predictions, references=labels),
    recall_metric.compute(predictions=predictions, references=labels),
    matthews_metric.compute(predictions=predictions, references=labels)


trainer = Trainer(
    model=model,
    args=arguments,
    train_dataset=small_tokenized_dataset['train'],
    eval_dataset=small_tokenized_dataset['val'], # change to test when you do your final evaluation!
    processing_class=distilbert_tokenizer, #cahnge it for your tokenizer from previous cells
    data_collator=data_collator,
    compute_metrics=compute_metrics
)